<a href="https://colab.research.google.com/github/Rumeysakeskin/TTS-turkish-emotion/blob/embed/evaluation_metrics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

UTMOS V2

PESQ

DNSMOS

STOI

NISQA

MCD

log-F0

WER

In [ ]:
# https://github.com/sarulab-speech/UTMOSv2

# https://github.com/sarulab-speech/UTMOSv2

!pip install -q git+https://github.com/sarulab-speech/UTMOSv2.git

import os
import glob
import pandas as pd
from tqdm import tqdm
import utmosv2

# =============================
# Model
# =============================
model = utmosv2.create_model(pretrained=True)

# =============================
# Directories
# =============================
DIRS = {
    "reference": "/content/drive/MyDrive/rumeysa-master-thesis/tez-evaluation/references",
    "xtts-finetune": "/content/drive/MyDrive/rumeysa-master-thesis/tez-evaluation/xtts-output-19-12-2025",
    "xtts-base": "/content/drive/MyDrive/rumeysa-master-thesis/tez-evaluation/base-output-19-12-2025",
    "mms": "/content/drive/MyDrive/rumeysa-master-thesis/tez-evaluation/mms-output-19-12-2025",
    "speech-t5": "/content/drive/MyDrive/rumeysa-master-thesis/tez-evaluation/speech-t5",
}

# =============================
# Eval function (INDEPENDENT)
# =============================
def eval_folder(folder_path, tag):
    wav_files = sorted(glob.glob(os.path.join(folder_path, "*.wav")))
    results = []

    print(f"\n🎧 Evaluating {tag}")
    print(f"Found {len(wav_files)} wav files")

    for wav_path in tqdm(wav_files):
        try:
            mos = model.predict(input_path=wav_path)
        except Exception as e:
            print(f"⚠️ Failed: {wav_path} | {e}")
            continue

        results.append({
            "folder": tag,
            "file": os.path.basename(wav_path),
            "utmosv2": float(mos)
        })

    return pd.DataFrame(results)

# =============================
# Run evaluation
# =============================
dfs = []

for tag, path in DIRS.items():
    dfs.append(eval_folder(path, tag))

df = pd.concat(dfs, ignore_index=True)
df

# =============================
# Summary statistics
# =============================
summary = (
    df.groupby("folder")["utmosv2"]
      .agg(["count", "mean", "std", "min", "max"])
      .reset_index()
)

summary


In [ ]:
!pip install torch torchaudio torchmetrics soundfile torchcodec
import os
from pathlib import Path

import torch
import torchaudio
from torchmetrics.audio import PerceptualEvaluationSpeechQuality

#    "reference": "/content/drive/MyDrive/rumeysa-master-thesis/tez-evaluation/references",
#    "base": "/content/drive/MyDrive/rumeysa-master-thesis/tez-evaluation/base-output-19-12-2025",
#    "output": "/content/drive/MyDrive/rumeysa-master-thesis/tez-evaluation/xtts-output-19-12-2025",

# =========================
# CONFIG
# =========================
REFERENCES_DIR = "/content/drive/MyDrive/rumeysa-master-thesis/tez-evaluation/references"
OUTPUTS_DIR = "/content/drive/MyDrive/rumeysa-master-thesis/tez-evaluation/xtts-output-19-12-2025"
SAMPLE_RATE = 16000
PESQ_MODE = "wb"  # wideband

# =========================
# INIT METRIC
# =========================
pesq = PerceptualEvaluationSpeechQuality(
    fs=SAMPLE_RATE,
    mode=PESQ_MODE
)

# =========================
# UTILS
# =========================
def load_wav(path, target_sr=16000):
    wav, sr = torchaudio.load(path)

    # mono
    if wav.shape[0] > 1:
        wav = wav.mean(dim=0, keepdim=True)

    # resample if needed
    if sr != target_sr:
        wav = torchaudio.functional.resample(wav, sr, target_sr)

    return wav.squeeze(0)


def match_length(a, b):
    """Pad or truncate to same length"""
    min_len = min(len(a), len(b))
    return a[:min_len], b[:min_len]


# =========================
# COLLECT FILES
# =========================
ref_files = {
    f.name: f for f in Path(REFERENCES_DIR).glob("*.wav")
}
out_files = {
    f.name: f for f in Path(OUTPUTS_DIR).glob("*.wav")
}

common_files = sorted(set(ref_files.keys()) & set(out_files.keys()))

if not common_files:
    raise RuntimeError("❌ Eşleşen .wav dosyası bulunamadı")

print(f"🔍 Found {len(common_files)} matching wav pairs")

# =========================
# EVALUATION LOOP
# =========================
scores = []

for fname in common_files:
    ref_path = ref_files[fname]
    out_path = out_files[fname]

    ref = load_wav(ref_path)
    deg = load_wav(out_path)

    ref, deg = match_length(ref, deg)

    score = pesq(deg, ref).item()
    scores.append(score)

    print(f"{fname:30s} PESQ: {score:.4f}")

# =========================
# SUMMARY
# =========================
scores = torch.tensor(scores)

print("\n==============================")
print("PESQ MODE: WB (16 kHz)")
print(f"Evaluated files: {len(scores)}")
print(f"Average PESQ: {scores.mean():.4f}")
print(f"Min PESQ:     {scores.min():.4f}")
print(f"Max PESQ:     {scores.max():.4f}")
print("==============================")



In [ ]:
!pip install torchcodec
!pip install -U torchmetrics[audio]

import torch
import torchaudio
from pathlib import Path
from torchmetrics.audio.dnsmos import DeepNoiseSuppressionMeanOpinionScore

# =========================
# CONFIG
# =========================
DIRS = {
    "reference": "/content/drive/MyDrive/rumeysa-master-thesis/tez-evaluation/references",
    "base": "/content/drive/MyDrive/rumeysa-master-thesis/tez-evaluation/base-output-19-12-2025",
    "output": "//content/drive/MyDrive/rumeysa-master-thesis/tez-evaluation/xtts-output-19-12-2025",
}


SAMPLE_RATE = 16000
PERSONALIZED = False

# =========================
# METRIC
# =========================
dnsmos = DeepNoiseSuppressionMeanOpinionScore(
    fs=SAMPLE_RATE,
    personalized=PERSONALIZED
)

# =========================
# UTILS
# =========================
def load_wav(path, target_sr=16000):
    wav, sr = torchaudio.load(path)

    # mono
    if wav.shape[0] > 1:
        wav = wav.mean(dim=0, keepdim=True)

    if sr != target_sr:
        wav = torchaudio.functional.resample(wav, sr, target_sr)

    return wav.squeeze(0)


# =========================
# EVALUATION
# =========================
results = {}

for label, dir_path in DIRS.items():
    scores = []

    wav_files = list(Path(dir_path).glob("*.wav"))
    if not wav_files:
        print(f"⚠️ No wav files in {label}")
        continue

    print(f"\n📁 Evaluating {label} ({len(wav_files)} files)")

    for wav_path in wav_files:
        audio = load_wav(wav_path)

        # DNSMOS → [p808, sig, bak, ovr]
        score = dnsmos(audio).cpu()
        scores.append(score)

        print(
            f"{wav_path.name:30s} "
            f"OVR={score[3]:.3f}  "
            f"SIG={score[1]:.3f}  "
            f"BAK={score[2]:.3f}"
        )

    scores = torch.stack(scores)
    results[label] = scores

    print("---- SUMMARY ----")
    print(f"Mean OVR: {scores[:,3].mean():.3f}")
    print(f"Std  OVR: {scores[:,3].std():.3f}")
    print(f"Min  OVR: {scores[:,3].min():.3f}")
    print(f"Max  OVR: {scores[:,3].max():.3f}")

# =========================
# FINAL COMPARISON
# =========================
print("\n==============================")
print("FINAL DNSMOS COMPARISON (OVR)")
print("==============================")

for label, scores in results.items():
    print(f"{label:10s} mean OVR: {scores[:,3].mean():.3f}")


In [ ]:
!pip install torch torchaudio torchmetrics[audio] pystoi torchcodec

import os
import torch
import torchaudio
from torchmetrics.audio import ShortTimeObjectiveIntelligibility


#    "reference": "/content/drive/MyDrive/rumeysa-master-thesis/tez-evaluation/references",
#    "base": "/content/drive/MyDrive/rumeysa-master-thesis/tez-evaluation/base-output-19-12-2025",
#    "output": "/content/drive/MyDrive/rumeysa-master-thesis/tez-evaluation/xtts-output-19-12-2025",


# ==============================
# CONFIG
# ==============================
REFERENCES_DIR = "/content/drive/MyDrive/rumeysa-master-thesis/tez-evaluation/references"
OUTPUTS_DIR = "/content/drive/MyDrive/rumeysa-master-thesis/tez-evaluation/xtts-output-19-12-2025"
SAMPLE_RATE = 16000   # WAV'lerin sample rate'i
EXTENDED = False      # True -> Extended STOI (eSTOI)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ==============================
# METRIC
# ==============================
stoi_metric = ShortTimeObjectiveIntelligibility(
    fs=SAMPLE_RATE,
    extended=EXTENDED
)

# ==============================
# HELPER
# ==============================
def load_wav(path, target_sr):
    wav, sr = torchaudio.load(path)
    wav = wav.mean(dim=0)  # mono
    if sr != target_sr:
        wav = torchaudio.functional.resample(wav, sr, target_sr)
    return wav

# ==============================
# EVALUATION LOOP
# ==============================
stoi_scores = []

for fname in sorted(os.listdir(REFERENCES_DIR)):
    ref_path = os.path.join(REFERENCES_DIR, fname)
    out_path = os.path.join(OUTPUTS_DIR, fname)

    if not os.path.exists(out_path):
        print(f"❌ Missing output for {fname}, skipping.")
        continue

    ref = load_wav(ref_path, SAMPLE_RATE)
    out = load_wav(out_path, SAMPLE_RATE)

    # Length alignment (CRITICAL for STOI)
    min_len = min(ref.shape[-1], out.shape[-1])
    ref = ref[:min_len]
    out = out[:min_len]

    score = stoi_metric(out, ref)
    stoi_scores.append(score.item())

    print(f"🔊 {fname} | STOI: {score.item():.4f}")

# ==============================
# FINAL RESULT
# ==============================
if stoi_scores:
    mean_stoi = sum(stoi_scores) / len(stoi_scores)
    print("\n==============================")
    print(f"✅ MEAN STOI: {mean_stoi:.4f}")
    print("==============================")
else:
    print("⚠️ No valid files evaluated.")


In [ ]:
!pip install torch torchaudio torchmetrics[audio] librosa requests torchcodec

import os
import torch
import torchaudio
from torchmetrics.functional.audio.nisqa import non_intrusive_speech_quality_assessment

# ==============================
# CONFIG
# ==============================
DIRS = {
    "reference": "/content/drive/MyDrive/rumeysa-master-thesis/tez-evaluation/references",
    "base": "/content/drive/MyDrive/rumeysa-master-thesis/tez-evaluation/base-output-19-12-2025",
    "output": "/content/drive/MyDrive/rumeysa-master-thesis/tez-evaluation/xtts-output-19-12-2025",
}

SAMPLE_RATE = 16000
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ==============================
# HELPERS
# ==============================
def load_wav(path, target_sr):
    wav, sr = torchaudio.load(path)
    wav = wav.mean(dim=0)  # mono
    if sr != target_sr:
        wav = torchaudio.functional.resample(wav, sr, target_sr)
    return wav

# ==============================
# EVALUATION
# ==============================
print("\n========== NISQA AVERAGE RESULTS ==========")

for tag, dir_path in DIRS.items():
    mos, noise, disc, color, loud = [], [], [], [], []
    count = 0

    for fname in sorted(os.listdir(dir_path)):
        if not fname.endswith(".wav"):
            continue

        wav_path = os.path.join(dir_path, fname)
        wav = load_wav(wav_path, SAMPLE_RATE)

        # Minimum length safety
        if wav.shape[-1] < SAMPLE_RATE:
            continue

        with torch.no_grad():
            scores = non_intrusive_speech_quality_assessment(
                wav.to(device),
                fs=SAMPLE_RATE
            ).cpu()

        mos.append(scores[0].item())
        noise.append(scores[1].item())
        disc.append(scores[2].item())
        color.append(scores[3].item())
        loud.append(scores[4].item())
        count += 1

    if count == 0:
        print(f"{tag.upper():10s} | ❌ No valid files")
        continue

    print(
        f"{tag.upper():10s} | "
        f"MOS: {sum(mos)/count:.3f} | "
        f"Noise: {sum(noise)/count:.3f} | "
        f"Disc: {sum(disc)/count:.3f} | "
        f"Color: {sum(color)/count:.3f} | "
        f"Loud: {sum(loud)/count:.3f} | "
        f"N={count}"
    )



In [ ]:
!pip install mel-cepstral-distance

import os
from mel_cepstral_distance import compare_audio_files
from tqdm import tqdm
import numpy as np

REFERENCE_DIR = "/content/drive/MyDrive/rumeysa-master-thesis/tez-evaluation/references"
BASE_DIR      = "/content/drive/MyDrive/rumeysa-master-thesis/tez-evaluation/base-output-19-12-2025"
OUTPUT_DIR    = "/content/drive/MyDrive/rumeysa-master-thesis/tez-evaluation/xtts-output-19-12-2025"


def evaluate_against_reference(reference_dir, target_dir):
    mcd_list = []
    penalty_list = []
    missing_files = []

    wav_files = sorted([
        f for f in os.listdir(reference_dir)
        if f.lower().endswith(".wav")
    ])

    for wav in tqdm(wav_files, desc=f"Evaluating {os.path.basename(target_dir)}"):
        ref_path = os.path.join(reference_dir, wav)
        target_path = os.path.join(target_dir, wav)

        if not os.path.exists(target_path):
            missing_files.append(wav)
            continue

        try:
            mcd, penalty = compare_audio_files(ref_path, target_path)
            mcd_list.append(mcd)
            penalty_list.append(penalty)
        except Exception as e:
            print(f"❌ Error processing {wav}: {e}")

    return {
        "count": len(mcd_list),
        "mcd_mean": np.mean(mcd_list),
        "mcd_std": np.std(mcd_list),
        "penalty_mean": np.mean(penalty_list),
        "penalty_std": np.std(penalty_list),
        "missing_files": missing_files
    }


# ==========================
# RUN EVALUATION
# ==========================

base_results = evaluate_against_reference(REFERENCE_DIR, BASE_DIR)
output_results = evaluate_against_reference(REFERENCE_DIR, OUTPUT_DIR)


# ==========================
# PRINT RESULTS
# ==========================

def print_results(name, results):
    print(f"\n📊 {name} RESULTS")
    print(f"Samples evaluated : {results['count']}")
    print(f"MCD Mean ± Std    : {results['mcd_mean']:.2f} ± {results['mcd_std']:.2f}")
    print(f"Penalty Mean ± Std: {results['penalty_mean']:.4f} ± {results['penalty_std']:.4f}")

    if results["missing_files"]:
        print(f"⚠ Missing files ({len(results['missing_files'])}):")
        for f in results["missing_files"]:
            print(f"  - {f}")


print_results("BASE vs REFERENCE", base_results)
print_results("XTTS OUTPUT vs REFERENCE", output_results)


In [ ]:
# =========================================================
# F0 RMSE (DTW + log-F0, voiced-only) for TTS Evaluation
# =========================================================

!pip install librosa pyworld numpy tqdm soundfile

import os
import librosa
import pyworld
import numpy as np
from tqdm import tqdm
from librosa.sequence import dtw

#    "reference": "/content/drive/MyDrive/rumeysa-master-thesis/tez-evaluation/references",
#    "base": "/content/drive/MyDrive/rumeysa-master-thesis/tez-evaluation/base-output-19-12-2025",
#    "output": "/content/drive/MyDrive/rumeysa-master-thesis/tez-evaluation/xtts-output-19-12-2025",

# -------------------------
# CONFIG
# -------------------------
REF_DIR = "/content/drive/MyDrive/rumeysa-master-thesis/tez-evaluation/references"
SYN_DIR = "/content/drive/MyDrive/rumeysa-master-thesis/tez-evaluation/base-output-19-12-2025"

SAMPLE_RATE = 16000
FRAME_PERIOD = 5.0  # ms

F0_FLOOR = 50.0
F0_CEIL  = 500.0

# -------------------------
# AUDIO LOADING
# -------------------------
def load_wav(path, sr):
    wav, _ = librosa.load(path, sr=sr, mono=True)
    return wav.astype(np.float64)

# -------------------------
# F0 EXTRACTION (WORLD)
# -------------------------
def extract_f0(wav, fs, frame_period):
    f0, _ = pyworld.harvest(
        wav,
        fs,
        frame_period=frame_period,
        f0_floor=F0_FLOOR,
        f0_ceil=F0_CEIL
    )
    return f0

# -------------------------
# LOG-F0 RMSE (VOICED ONLY)
# -------------------------
def log_f0_rmse(f0_ref, f0_syn):
    mask = np.logical_and(f0_ref > 0, f0_syn > 0)

    if np.sum(mask) == 0:
        return None

    log_ref = np.log(f0_ref[mask])
    log_syn = np.log(f0_syn[mask])

    return np.sqrt(np.mean((log_ref - log_syn) ** 2))

# -------------------------
# MAIN EVALUATION
# -------------------------
def evaluate_f0_rmse(ref_dir, syn_dir):
    scores = []

    files = sorted([
        f for f in os.listdir(ref_dir)
        if f.endswith(".wav") and os.path.exists(os.path.join(syn_dir, f))
    ])

    print(f"Found {len(files)} matching wav files")

    if len(files) == 0:
        return None

    for fname in tqdm(files):
        ref_path = os.path.join(ref_dir, fname)
        syn_path = os.path.join(syn_dir, fname)

        try:
            wav_ref = load_wav(ref_path, SAMPLE_RATE)
            wav_syn = load_wav(syn_path, SAMPLE_RATE)

            f0_ref = extract_f0(wav_ref, SAMPLE_RATE, FRAME_PERIOD)
            f0_syn = extract_f0(wav_syn, SAMPLE_RATE, FRAME_PERIOD)

            # ---- CRITICAL FIX: correct DTW shape ----
            f0_ref_dtw = f0_ref.reshape(1, -1)
            f0_syn_dtw = f0_syn.reshape(1, -1)

            _, wp = dtw(f0_ref_dtw, f0_syn_dtw)
            wp = np.array(wp)

            f0_ref_aligned = f0_ref[wp[:, 0]]
            f0_syn_aligned = f0_syn[wp[:, 1]]

            rmse = log_f0_rmse(f0_ref_aligned, f0_syn_aligned)

            if rmse is not None:
                scores.append(rmse)

        except Exception as e:
            print(f"[ERROR] {fname}: {e}")

    return np.array(scores)

# -------------------------
# RUN
# -------------------------
if __name__ == "__main__":
    scores = evaluate_f0_rmse(REF_DIR, SYN_DIR)

    if scores is not None and len(scores) > 0:
        print("\n" + "=" * 60)
        print("F0 RMSE RESULTS (DTW + log-F0, LOWER IS BETTER)")
        print(f"Evaluated files : {len(scores)}")
        print(f"Mean F0 RMSE   : {scores.mean():.4f}")
        print(f"Min  F0 RMSE   : {scores.min():.4f}")
        print(f"Max  F0 RMSE   : {scores.max():.4f}")
        print("=" * 60)
    else:
        print("\n❌ No valid F0 RMSE scores computed")


In [ ]:
references = {
    "audio1.wav": "Ancak yaratığın adı basit kalırken itibarı kısa sürede oldukça karmaşık hale geldi.",
    "audio2.wav": "Bir limonatacıda beş dakika oturmaya razı oldu ve hikayesine orada devam etti.",
    "audio3.wav": "Aslında bunu yapmaktaki amaç kölelik ve onun etrafında şekillenen dilin günümüzde hala bizim hayatımızın bir parçası olduğunu göstermektedir.",
    "audio4.wav": "Bankın tepesinde ellerin neredeyse kavuşuyor olduğu hariç ikisi arasında bir bariyer gibi kullanılmış olması detayı çok hoş.",
    "audio5.wav": "Reel faiz oranı ve reel gayri safi hasılı açısından dengede dedik. Şimdi bir düşünelim. Merkez bankası fazladan para basmaya kalkarsa ne olur?",
    "audio6.wav": "Hızlı konuşma problemi nasıl çözülür? Bununla ilgili birkaç tane tavsiye verdim. Çok hızlı konuşan insanlar genelde çok hızlı düşünen insanlardır.",
    "audio7.wav": "Son olarak ta şuralara inerseniz fiyatlarımız oldukça aşağıda olacak. Fiyatlar oldukça aşağıda. Bu durumda bir dolarlık bir değişim çok büyük yüzdesel fiyat değişimi değil mi?",
    "audio8.wav": "Yine fiziki altınlarımızda altın parçalarıyla herhangi bir işlem yapmadık. Hemen akla şu soru gelebilir. Bu işlemler nereye kadar devam edebilir?",
    "audio9.wav": "Tarih boyunca sanat, büyük soruları cevaplamaya çalışmış ve burada gördüğümüz sanatsal çalışma da bu önemli soruları sormaya devam ediyor.",
    "audio10.wav": "Veya film rulosu kabınız yoksa, kendi yaptığınız siyah kabınızı sabunlu suya batırıp beyaz kağıdın üstüne koyabilirsiniz.",
    "audio11.wav": "Kırmızı ampulden çıkan ışık bu şekilde tahtaya ilerliyor, ama kalem onu bu şekilde engelliyor.",
    "audio12.wav": "Yine algılanabilir hale getirmek için biz bu resmin neresindeyiz göstereyim.",
    "audio13.wav": "Rutin hayattan alınmış, yükselebilen ve alçalabilen, çelişkiden uzak, göz kamaştırıcı bir tecrübe.",
    "audio14.wav": "Dolayısıyla da bu dönemden günümüze ulaşabilen bronz heykel sayısı son derece az, burada gördüğümüz de o nadir örneklerden birisi. Şu an görmekte olduğumuz, Milattan Önce 100 yıllarından kalmış bir heykel. 'Dinlenen Boksör' heykeline bakıyoruz.",
    "audio15.wav": "Bize insan doğası hakkında da çok şey söylüyor, bu sürekli ve daimi güzellik arayışına dair. Zaman geçtikçe parlaklığı azalabilir ama bu arayış hiç solmayacak.",
    "audio16.wav": "Bu insanların, yani Atinalıların vahşi doğayı kontrol edecek güce sahip olduğu simgeleniyor bu frizede. Atlar vahşi doğayı temsil ediyor.",
    "audio17.wav": "Hücre dışındaki bu pozitif yükler kendileriyle aynı yükten olan diğer iyonlardan uzaklaşmak ve daha negatif yüklü olan tarafa doğru hareket etmek isteyecekler.",
    "audio18.wav": "Eğer 100 derecede 1 gram su buharı varsa ve bunu yoğunlaştırmak istersem, sistemden bu kadar enerji almam gerekir.",
    "audio19.wav": "Kültürün bu iki cephesinin birbirine tamamen zıt olması dolayısıyla, yeni bir teknolojinin kabul edilmesi de zor olur.",
    "audio20.wav": "Önceki videoda galiba son olarak Güneş'in Dünya'ya göre ne kadar büyük olduğunu ve Dünya'nın Güneş'ten ne kadar uzak olduğunu görmüştük.",
}

audio_files = [
    "audio1.wav",
    "audio2.wav",
    "audio3.wav",
    "audio4.wav",
    "audio5.wav",
    "audio6.wav",
    "audio7.wav",
    "audio8.wav",
    "audio9.wav",
    "audio10.wav",
    "audio11.wav",
    "audio12.wav",
    "audio13.wav",
    "audio14.wav",
    "audio15.wav",
    "audio16.wav",
    "audio17.wav",
    "audio18.wav",
    "audio19.wav",
    "audio20.wav",
]

In [ ]:
!pip install -U openai-whisper
!pip install jiwer

import whisper
import os

#base_path = "/content/drive/MyDrive/rumeysa-master-thesis/tez-evaluation/references"
#base_path = "/content/drive/MyDrive/rumeysa-master-thesis/tez-evaluation/xtts-output-19-12-2025"
base_path = "/content/drive/MyDrive/rumeysa-master-thesis/tez-evaluation/base-output-19-12-2025"


audio_files = [os.path.join(base_path, f) for f in audio_files]

references = {
    os.path.join(base_path, k): v
    for k, v in references.items()
}

model = whisper.load_model("large-v3-turbo")

predictions = {}

for audio in audio_files:
    result = model.transcribe(
        audio,
        language="tr",
        task="transcribe",
        fp16=True
    )
    predictions[audio] = result["text"]
    print(f"{audio} → {result['text']}")


from jiwer import wer
import numpy as np

wers = {}

for audio in audio_files:
    ref = references[audio]
    hyp = predictions[audio]

    w = wer(ref, hyp)
    wers[audio] = w

    print(f"{audio} | WER: {w:.4f}")

average_wer = np.mean(list(wers.values()))

print("\n========== WER RESULTS ==========")
for k, v in wers.items():
    print(f"{k:12s} : {v:.4f}")

print(f"\nAVERAGE WER : {average_wer:.4f}")
